In [3]:
from chromadb.utils import embedding_functions
import chromadb
from tqdm import tqdm

Connect to Docker container hosting Chroma

In [4]:
client = chromadb.HttpClient(host='localhost', port=8000) #connect to server on Docker container

Create a collection that utilizes BeRT embedding (via calls to HuggingFace).

In [5]:
from chromadb.utils import embedding_functions
 
# Define the embedding model (all-MiniLM-L6-v2 is fast and efficient for this scenario)
default_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
 
# Get our existing collection and assign this embedding function to it
hotel_collection = client.get_or_create_collection(
    name="hotel_descriptions",
    embedding_function=default_ef
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Raw data to chunk and embed.

In [6]:
# Our raw data from the travel agency

raw_hotel_data = [ {"id": "h1", "description": "Peaceful woodland cabin. Features a stone fireplace and floor-to-ceiling forest views. Located in the heart of the Smokies.", "price": 150}, {"id": "h2", "description": "Modern downtown apartment. Steps away from the subway, nightlife, and high-end shopping districts.", "price": 250}, {"id": "h3", "description": "Luxury beach resort. Includes private infinity pools, white sand ocean access, and 24/7 room service.", "price": 500}, {"id": "h4", "description": "Quiet forest retreat. Tucked away in dense pine trees with minimalist decor and no Wi-Fi for total disconnection.", "price": 120} ]


Chunk by sentence ('.' delimiter), and add each chunk to database with unique ID. This code can be modified to also add metadata for each description.

In [7]:
documents = []; metadatas = []; ids = [];
id=0;
for hotel in raw_hotel_data: # We split by the period to create "chunks"
    chunks = hotel['description'].split('. ') 
    for i, chunk in enumerate(chunks): 
        documents.append(chunk) # We keep the original hotel ID and price as metadata for filtering later metadatas.append({"hotel_id": hotel['id'], "price": hotel['price']}) ids.append(f"{hotel['id']}chunk{i}")
        ids.append(str(id))
        id=id+1
print(f"Prepared {len(documents)} chunks from {len(raw_hotel_data)} hotels.")


Prepared 9 chunks from 4 hotels.


In [8]:
hotel_collection.add(documents=documents, ids=ids)
print(f"Successfully embedded and stored {hotel_collection.count()} vectors.")


Successfully embedded and stored 11 vectors.


In [9]:
sample = hotel_collection.get(include=['embeddings', 'documents'], limit=1) 
print(f"Document: {sample['documents'][0]}") 
print(f"Vector Preview (first 5 dimensions): {sample['embeddings'][0][:5]}")
print(f"Length of embedding vector: {len(sample['embeddings'][0])}")

Document: Peaceful woodland cabin
Vector Preview (first 5 dimensions): [ 0.10545908  0.04965884 -0.03755431  0.0574775   0.07096738]
Length of embedding vector: 384


In [10]:
# Adding a single new entry

hotel_collection.add( documents=["Eco-friendly mountain yurt with 360-degree alpine views and solar power."], metadatas=[{"hotel_id": "h5", "price": 90}], ids=["h2_chunk_0"])
print(f"New total count: {hotel_collection.count()}")


New total count: 12


In [11]:
#Reading a specific entry by its ID

entry = hotel_collection.get(ids=["0"])
print("Retrieved Document:", entry['documents'][0]) 
print("Metadata:", entry['metadatas'][0])
#Advanced Read: Filtering by metadata (e.g., all hotels under $130)
budget_hotels = hotel_collection.get( where={"price": {"$lt": 130}} ) 
print(f"Found {len(budget_hotels['ids'])} budget options.")



Retrieved Document: Peaceful woodland cabin
Metadata: None
Found 3 budget options.


In [12]:
# Updating the text and metadata for the downtown apartment
    
hotel_collection.update( ids="h2_chunk_0", documents=["Ultra-modern penthouse suite with rooftop pool and city skyline views.",], metadatas=[{"hotel_id": "h2", "price": 450}] )
#Verify the update
updated_entry = hotel_collection.get(ids=["h2_chunk_0"]) 
print("Updated Description:", updated_entry['documents'][0])


Updated Description: Ultra-modern penthouse suite with rooftop pool and city skyline views.


In [13]:
#Deleting the 'h4' forest retreat chunks
hotel_collection.delete( ids=["h2_chunk_0"] )
print(f"Final collection count after deletion: {hotel_collection.count()}")


Final collection count after deletion: 11


In [ ]:
# The "Keyword" search would fail here because of no exact matches for terms, but Semantic Search wins.

query = "I need a quiet cottage in a secluded area."
results = hotel_collection.query( query_texts=[query], n_results=1, include=['distances', 'documents'] )
# Distance in Chroma (using 'cosine' space) is often (1 - similarity)
# So a smaller distance = a better match!
print(f"Top Match Found: {results['documents'][0][0]}") 
print(f"Mathematical Distance: {results['distances'][0][0]:.4f}")


Top Match Found: Peaceful woodland cabin
Mathematical Distance: 0.4542


# Challenge:

Return 7 query results and sort by distance

In [ ]:
#Right answer

# Perform the query
results = hotel_collection.query(
    query_texts=["I need a quiet cottage in a secluded area."],
    n_results=7,
    include=["documents", "distances", "metadatas"]
)

# The results are already sorted by distance
for i in range(len(results["ids"][0])):
    print(f"Rank {i+1}")
    print(f"Distance: {results['distances'][0][i]}")
    print(f"Document: {results['documents'][0][i][:10]}...\n")

Rank 1
Distance: 0.45422798
Document: Peaceful woodland cabin...

Rank 2
Distance: 0.47504905
Document: Quiet forest retreat...

Rank 3
Distance: 0.60278296
Document: Tucked away in dense pine trees with minimalist decor and no Wi-Fi for total disconnection....

Rank 4
Distance: 0.62272483
Document: Luxury beach resort...

Rank 5
Distance: 0.6910681
Document: Eco-friendly mountain yurt with 360-degree alpine views and solar power....

Rank 6
Distance: 0.6910681
Document: Eco-friendly mountain yurt with 360-degree alpine views and solar power....

Rank 7
Distance: 0.7025901
Document: Modern downtown apartment...



### Wrong Answers

`results = hotel_collection.query(
    query_texts=["I need a quiet cottage in a secluded area."],
    include=["documents", "distances", "metadatas", n_results=7]
)`
#### Hint: Remember to specify the number of results returned

`results = hotel_collection.query(
    query_texts=["I need a quiet cottage in a secluded area. Include top 7 results and the documents, distances, and metadatas."]`
) 
#### Hint: You can't just include the instructions in the prompt (unlike with direct calls to an LLM)

`results = hotel_collection.query(
    query_texts=["I need a quiet cottage in a secluded area."],
    params = {'n_results' : '7'}
    include=["documents", "distances", "metadatas"]
)`
#### Hint: Double check the function call signature